# Upstream NISAR product discovery handoff

This optional example keeps product acquisition outside SnowIn:

```text
nisar-pytools → search / download / validate → local GUNW path → SnowIn
```

Install `nisar-pytools` (included in SnowIn's `nisar` extra) and the `notebooks` dependency group. Remote search and product download are both disabled in the configuration cell. Set `RUN_REMOTE_SEARCH = True` to list ASF products; set `DOWNLOAD_PRODUCT_INDEX` to one listed index only when you intend to download it. ASF access may require locally configured Earthdata credentials. The GUNW is saved under `data/external/nisar/gunw/` for notebook 02.

In [ ]:
from pathlib import Path

from nisar_pytools import download_urls, find_nisar

# Set these explicitly before running remote acquisition.
RUN_REMOTE_SEARCH = False
DOWNLOAD_PRODUCT_INDEX = None  # Keep None to search only.

working_path = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (working_path, *working_path.parents)
        if (candidate / "src" / "snowin").is_dir()
    ),
    working_path,
)
output_dir = repo_root / "data" / "external" / "nisar" / "gunw"

if RUN_REMOTE_SEARCH:
    aoi = [
        float(value)
        for value in input("AOI west,south,east,north: ").split(",")
    ]
    if len(aoi) != 4:
        raise ValueError("Enter four AOI bounds: west,south,east,north")
    start_date = input("Start date (YYYY-MM-DD): ").strip()
    end_date = input("End date (YYYY-MM-DD): ").strip()
else:
    aoi = None
    start_date = None
    end_date = None

## 1. Search upstream

`find_nisar` is called directly from `nisar-pytools`. This cell makes no remote request unless `RUN_REMOTE_SEARCH` is `True`. Leave `DOWNLOAD_PRODUCT_INDEX` as `None` to use this as a search-only dry run.

In [ ]:
if RUN_REMOTE_SEARCH:
    urls = find_nisar(
        aoi=aoi,
        start_date=start_date,
        end_date=end_date,
        product_type="GUNW",
        max_results=20,
    )
    print(f"Found {len(urls)} GUNW products")
    for index, url in enumerate(urls):
        print(index, url.rsplit("/", 1)[-1])
else:
    urls = []
    print("Remote search is disabled. Set RUN_REMOTE_SEARCH = True to search ASF.")

## 2. Select, download, and validate one product

A product is downloaded only when `DOWNLOAD_PRODUCT_INDEX` is an integer and remote search is enabled. The validated GUNW is saved in `data/external/nisar/gunw/`, the same location used by the command-line script and notebook 02.

In [ ]:
if DOWNLOAD_PRODUCT_INDEX is None:
    print(
        "Product download is disabled. Set DOWNLOAD_PRODUCT_INDEX after "
        "reviewing the search results."
    )
elif not RUN_REMOTE_SEARCH:
    raise ValueError("Set RUN_REMOTE_SEARCH = True before enabling a product download.")
elif not urls:
    raise RuntimeError("No GUNW products matched the search.")
elif DOWNLOAD_PRODUCT_INDEX not in range(len(urls)):
    raise IndexError("Select one of the listed product indices.")
else:
    output_dir.mkdir(parents=True, exist_ok=True)
    files = download_urls(
        [urls[DOWNLOAD_PRODUCT_INDEX]], output_dir, max_workers=1, validate=True
    )
    if not files:
        raise RuntimeError("No product was downloaded and validated.")
    gunw_path = Path(files[0]).expanduser().resolve()
    if not gunw_path.is_file():
        raise FileNotFoundError(gunw_path)
    print("Validated local GUNW:", gunw_path)
    print(
        "Set SNOWIN_GUNW to this path if data/external/nisar/gunw/ "
        "contains multiple products."
    )
    print(
        "For terrain-local incidence, also put the matching "
        "NISAR-modified Copernicus DEM in data/external/nisar/dem/."
    )